In [1]:
import json
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize
import pandas as pd
from tqdm import tqdm

In [2]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\tomci\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\tomci\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\tomci\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\tomci\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\tomci\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\tomci\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:
TARGET_DIR = "data/"
PARQUET_FILE = TARGET_DIR + "wiki_literature_preprocessed.parquet"
CSV_FILE = TARGET_DIR + "wiki_literature_preprocessed.csv"

RAW_DATA_DIR = "data/scraping/"
RAW_DATA_FILE = RAW_DATA_DIR + "scraped_data.jsonl"

DOMAIN_STOPWORDS = {
    "isbn",
    "oclc",
    "doi",
    "issn",
    "page",
    "pages",
    "volume",
    "vol",
    "retrieved",
    "archived",
    "original",
    "cite",
    "citation",
    "press",
    "edition",
    "editions",
    "publisher",
    "published",
    "publication",
    "chapter",
    "author",
    "literary",
    "literature",
    "also",
    "often",
    "would",
    "one",
    "first",
    "two",
    "new",
    "well",
    "known",
    "may",
    "many",
    "see",
}

save_columns = [
    "title",
    "url",
    "summary",
    "categories",
    "text_lemmatized",
    "text_stemmed",
    "token_count",
    "vocab_size",
]

In [4]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

stop_words = set(stopwords.words("english"))
stop_words.update(DOMAIN_STOPWORDS)

In [5]:
def clean_raw_text(text: str) -> str:
    """Lowercases, removes URLs, numbers, special characters and extra spaces."""
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text) # Remove URLs
    text = re.sub(r"[^a-z\s-]", " ", text) # Remove numbers & special characters
    text = text.replace("-", " ")
    text = re.sub(r"\s+", " ", text).strip() # Remove extra whitespace
    return text

In [6]:
def preprocess_text(text: str) -> dict:
    """Performs tokenization, stopword removal, stemming, and lemmatization."""
    cleaned = clean_raw_text(text)
    tokens = word_tokenize(cleaned)

    filtered_tokens = [w for w in tokens if w not in stop_words and len(w) >= 3]

    lemmatized = [lemmatizer.lemmatize(w) for w in filtered_tokens]
    stemmed = [stemmer.stem(w) for w in filtered_tokens]

    return {
        "tokens_clean": filtered_tokens,
        "tokens_lemmatized": lemmatized,
        "tokens_stemmed": stemmed,
        "text_lemmatized": " ".join(lemmatized),
        "text_stemmed": " ".join(stemmed),
        "vocab_size": len(set(lemmatized)),
        "token_count": len(filtered_tokens)
    }

In [ ]:
def read_from_file():
    records = []
    with open(RAW_DATA_FILE, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} records from {RAW_DATA_FILE}")
    
    return df

In [8]:
def process_df(df):
    tqdm.pandas(desc="Preprocessing Documents")
    processed_results = df["text"].progress_apply(preprocess_text)

    processed_df = pd.DataFrame(list(processed_results))
    df = pd.concat([df, processed_df], axis=1)

    initial_count = len(df)
    df = df[df["token_count"] >= 30].reset_index(drop=True)
    print(f"Retained {len(df)}/{initial_count} documents with sufficient text.")

    return df

In [9]:
def save_to_files(df, save_columns):
    df[save_columns].to_parquet(PARQUET_FILE, index=False)

    csv_df = df[save_columns].copy()
    csv_df["categories"] = csv_df["categories"].apply(lambda cats: "|".join(cats))
    csv_df.to_csv(CSV_FILE, index=False, encoding="utf-8")

In [10]:
def main():
    df = read_from_file()
    df = process_df(df)
    save_to_files(df, save_columns)

In [11]:
main()

Loaded 1079 records from scraped_data.jsonl


Preprocessing Documents: 100%|██████████| 1079/1079 [01:10<00:00, 15.39it/s]


Retained 1076/1079 documents with sufficient text.
